In [17]:
import plotly.graph_objects as go

In [18]:
import plotly.graph_objects as go
import plotly.express as px
import pandas as pd
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output


# ============================================================
# 1. LOAD DATASET
# ============================================================

df = pd.read_csv("D:/pp/E-Commerce Sales Analytics.csv")


# ============================================================
# 2. DATA CLEANING
# ============================================================

df['revenue'] = pd.to_numeric(
    df['revenue'],
    errors='coerce'
)

df['quantity'] = pd.to_numeric(
    df['quantity'],
    errors='coerce'
)

df['customer_rating'] = pd.to_numeric(
    df['customer_rating'],
    errors='coerce'
)

df = df.dropna(
    subset=[
        'product_category',
        'region',
        'payment_method',
        'revenue',
        'quantity',
        'customer_rating',
        'order_id'
    ]
)


# ============================================================
# 3. CREATE SUMMARY DATA
# ============================================================

summary_df = (
    df.groupby(
        [
            'region',
            'product_category',
            'payment_method'
        ],
        as_index=False
    )
    .agg(
        Revenue=('revenue', 'sum'),
        Quantity=('quantity', 'sum'),
        Total_Orders=('order_id', 'nunique'),
        Customer_Rating=('customer_rating', 'mean')
    )
)


# ============================================================
# 4. REGION LIST
# ============================================================

regions = sorted(
    df['region'].dropna().unique()
)


# ============================================================
# 5. OUTPUT CONTAINERS
# ============================================================

sunburst_output = widgets.Output()
revenue_output = widgets.Output()
quantity_output = widgets.Output()
payment_output = widgets.Output()
rating_output = widgets.Output()
orders_output = widgets.Output()


# ============================================================
# 6. REGION DROPDOWN
# ============================================================

current_region = widgets.Dropdown(

    options=[
        'Select a Region...'
    ] + regions,

    value='Select a Region...',

    description='⚡ Region:',

    style={
        'description_width': 'initial'
    },

    layout=widgets.Layout(
        width='95%',
        margin='10px 0px'
    )
)


# ============================================================
# 7. DASHBOARD UPDATE FUNCTION
# ============================================================

def update_dashboard(change):

    selected_region = (
        change['new']
        if isinstance(change, dict)
        else change
    )


    # ========================================================
    # 1. SUNBURST
    # REGION → CATEGORY → PAYMENT METHOD
    # ========================================================

    with sunburst_output:

        clear_output(wait=True)

        fig_sunburst = px.sunburst(

            summary_df,

            path=[
                'region',
                'product_category',
                'payment_method'
            ],

            values='Revenue',

            color='region',

            color_discrete_sequence=
            px.colors.qualitative.Pastel

        )

        fig_sunburst.update_traces(

            textinfo='label+percent entry',

            hovertemplate=
            '<b>%{label}</b><br>' +
            'Revenue: ₹%{value:,.2f}<br>' +
            'Share: %{percentEntry:.1%}' +
            '<extra></extra>'

        )

        fig_sunburst.update_layout(

            title=
            "🎯 Region → Product Category → Payment Method",

            height=420,

            margin=dict(
                t=50,
                l=0,
                r=0,
                b=0
            )

        )

        fig_sunburst.show()


    # ========================================================
    # REGION SELECTED
    # ========================================================

    if selected_region in regions:

        filtered_df = summary_df[
            summary_df['region'] == selected_region
        ]


        # ====================================================
        # 2. REVENUE BY PRODUCT CATEGORY
        # ====================================================

        with revenue_output:

            clear_output(wait=True)

            category_revenue = (

                filtered_df
                .groupby(
                    'product_category',
                    as_index=False
                )
                ['Revenue']
                .sum()
                .sort_values(
                    'Revenue',
                    ascending=False
                )
            )


            fig_revenue = go.Figure(

                go.Bar(

                    x=category_revenue[
                        'product_category'
                    ],

                    y=category_revenue[
                        'Revenue'
                    ],

                    marker_color='royalblue',

                    text=[
                        f"₹{x:,.0f}"
                        for x in category_revenue[
                            'Revenue'
                        ]
                    ],

                    textposition='auto',

                    hovertemplate=
                    '<b>%{x}</b><br>' +
                    'Revenue: ₹%{y:,.2f}' +
                    '<extra></extra>'

                )

            )


            fig_revenue.update_layout(

                title=
                f"💰 Revenue by Product Category — "
                f"{selected_region}",

                xaxis_title=
                "Product Category",

                yaxis_title=
                "Total Revenue",

                height=380,

                margin=dict(
                    t=50,
                    l=40,
                    r=20,
                    b=40
                )

            )

            fig_revenue.show()


        # ====================================================
        # 3. QUANTITY BY PRODUCT CATEGORY
        # ====================================================

        with quantity_output:

            clear_output(wait=True)

            category_quantity = (

                filtered_df
                .groupby(
                    'product_category',
                    as_index=False
                )
                ['Quantity']
                .sum()
                .sort_values(
                    'Quantity',
                    ascending=False
                )
            )


            fig_quantity = go.Figure(

                go.Bar(

                    x=category_quantity[
                        'product_category'
                    ],

                    y=category_quantity[
                        'Quantity'
                    ],

                    marker_color='seagreen',

                    text=[
                        f"{x:,.0f}"
                        for x in category_quantity[
                            'Quantity'
                        ]
                    ],

                    textposition='auto',

                    hovertemplate=
                    '<b>%{x}</b><br>' +
                    'Quantity: %{y:,.0f}' +
                    '<extra></extra>'

                )

            )


            fig_quantity.update_layout(

                title=
                f"📦 Quantity Sold by Category — "
                f"{selected_region}",

                xaxis_title=
                "Product Category",

                yaxis_title=
                "Total Quantity",

                height=380,

                margin=dict(
                    t=50,
                    l=40,
                    r=20,
                    b=40
                )

            )

            fig_quantity.show()


        # ====================================================
        # 4. PAYMENT METHOD REVENUE
        # ====================================================

        with payment_output:

            clear_output(wait=True)

            payment_revenue = (

                filtered_df
                .groupby(
                    'payment_method',
                    as_index=False
                )
                ['Revenue']
                .sum()
                .sort_values(
                    'Revenue',
                    ascending=False
                )
            )


            fig_payment = go.Figure(

                go.Bar(

                    x=payment_revenue[
                        'payment_method'
                    ],

                    y=payment_revenue[
                        'Revenue'
                    ],

                    marker_color='orange',

                    text=[
                        f"₹{x:,.0f}"
                        for x in payment_revenue[
                            'Revenue'
                        ]
                    ],

                    textposition='auto',

                    hovertemplate=
                    '<b>%{x}</b><br>' +
                    'Revenue: ₹%{y:,.2f}' +
                    '<extra></extra>'

                )

            )


            fig_payment.update_layout(

                title=
                f"💳 Revenue by Payment Method — "
                f"{selected_region}",

                xaxis_title=
                "Payment Method",

                yaxis_title=
                "Total Revenue",

                height=380,

                margin=dict(
                    t=50,
                    l=40,
                    r=20,
                    b=40
                )

            )

            fig_payment.show()


        # ====================================================
        # 5. CUSTOMER RATING
        # ====================================================

        with rating_output:

            clear_output(wait=True)

            category_rating = (

                filtered_df
                .groupby(
                    'product_category',
                    as_index=False
                )
                ['Customer_Rating']
                .mean()
                .sort_values(
                    'Customer_Rating',
                    ascending=False
                )
            )


            fig_rating = go.Figure(

                go.Bar(

                    x=category_rating[
                        'product_category'
                    ],

                    y=category_rating[
                        'Customer_Rating'
                    ],

                    marker_color='mediumpurple',

                    text=[
                        f"{x:.2f}"
                        for x in category_rating[
                            'Customer_Rating'
                        ]
                    ],

                    textposition='auto',

                    hovertemplate=
                    '<b>%{x}</b><br>' +
                    'Average Rating: %{y:.2f}' +
                    '<extra></extra>'

                )

            )


            fig_rating.update_layout(

                title=
                f"⭐ Average Customer Rating — "
                f"{selected_region}",

                xaxis_title=
                "Product Category",

                yaxis_title=
                "Average Customer Rating",

                yaxis=dict(
                    range=[0, 5]
                ),

                height=380,

                margin=dict(
                    t=50,
                    l=40,
                    r=20,
                    b=40
                )

            )

            fig_rating.show()


        # ====================================================
        # 6. TOTAL ORDERS BY PRODUCT CATEGORY
        # ====================================================

        with orders_output:

            clear_output(wait=True)

            category_orders = (

                filtered_df
                .groupby(
                    'product_category',
                    as_index=False
                )
                ['Total_Orders']
                .sum()
                .sort_values(
                    'Total_Orders',
                    ascending=False
                )
            )


            fig_orders = go.Figure(

                go.Bar(

                    x=category_orders[
                        'product_category'
                    ],

                    y=category_orders[
                        'Total_Orders'
                    ],

                    marker_color='crimson',

                    text=[
                        f"{x:,.0f}"
                        for x in category_orders[
                            'Total_Orders'
                        ]
                    ],

                    textposition='auto',

                    hovertemplate=
                    '<b>%{x}</b><br>' +
                    'Total Orders: %{y:,.0f}' +
                    '<extra></extra>'

                )

            )


            fig_orders.update_layout(

                title=
                f"🛒 Total Orders by Product Category — "
                f"{selected_region}",

                xaxis_title=
                "Product Category",

                yaxis_title=
                "Total Orders",

                height=380,

                margin=dict(
                    t=50,
                    l=40,
                    r=20,
                    b=40
                )

            )

            fig_orders.show()


    # ========================================================
    # NO REGION SELECTED
    # ========================================================

    else:

        with revenue_output:

            clear_output(wait=True)

            print("\n" * 4)

            print(
                "        💡 Select a Region above"
            )

            print(
                "           to generate the detailed charts."
            )


        with quantity_output:
            clear_output(wait=True)

        with payment_output:
            clear_output(wait=True)

        with rating_output:
            clear_output(wait=True)

        with orders_output:
            clear_output(wait=True)


# ============================================================
# 8. EVENT BINDING
# ============================================================

current_region.observe(
    update_dashboard,
    names='value'
)


# ============================================================
# 9. DASHBOARD LAYOUT
# ============================================================

control_bar = widgets.VBox(
    [
        current_region
    ]
)


# ============================================================
# ROW 1
# ============================================================

row_1 = widgets.HBox(

    [
        sunburst_output,
        revenue_output
    ],

    layout=widgets.Layout(
        width='100%',
        align_items='center'
    )

)


# ============================================================
# ROW 2
# ============================================================

row_2 = widgets.HBox(

    [
        quantity_output,
        payment_output
    ],

    layout=widgets.Layout(
        width='100%',
        align_items='center'
    )

)


# ============================================================
# ROW 3
# ============================================================

row_3 = widgets.HBox(

    [
        rating_output,
        orders_output
    ],

    layout=widgets.Layout(
        width='100%',
        align_items='center'
    )

)


# ============================================================
# COMPLETE WORKSPACE
# ============================================================

full_workspace = widgets.VBox(

    [
        control_bar,
        row_1,
        row_2,
        row_3
    ],

    layout=widgets.Layout(
        width='100%'
    )

)


# ============================================================
# DISPLAY
# ============================================================

display(full_workspace)


# ============================================================
# INITIAL RENDER
# ============================================================

update_dashboard(
    current_region.value
)